# Semantic Similarity Metrics

String comparison breaks down when multiple phrasings mean the same thing. Semantic similarity measures how close two pieces of text are in meaning, not in characters. This notebook covers the `dspy.Embedder` interface, cosine similarity as a metric, multi-field averaging, and a pickle-backed cache that avoids re-embedding the same text twice.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## The Embedder

An embedding is a dense vector representation of text. Two texts that mean the same thing produce similar vectors. DSPy provides an `Embedder` interface that wraps the major embedding APIs.

In [ ]:
embedder = dspy.Embedder(
    "openai/text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Embed a single text
text = "The cat sat on the mat"
embedding = embedder([text])[0]

print(type(embedding))  # list of numbers
print(len(embedding))   # 1536 dimensions

## Cosine similarity

Once you have embeddings, measure their distance with cosine similarity. Think of embeddings as arrows in space — cosine similarity measures the angle between them, not the magnitude. Values range from -1 (opposite) to 1 (identical); above ~0.8 typically indicates strong semantic match.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


embedding1 = embedder(["The cat sat on the mat"])[0]
embedding2 = embedder(["A feline rested on the rug"])[0]
embedding3 = embedder(["The stock market crashed today"])[0]

print(cosine_similarity(embedding1, embedding2))  # ~0.85 (very similar)
print(cosine_similarity(embedding1, embedding3))  # ~0.45 (not similar)

## Using embeddings as a DSPy metric

Wrap the embedder and cosine similarity into a metric function that returns `True` when the predicted answer is within a similarity threshold of the gold answer.

In [ ]:
def semantic_similarity_metric(example, pred, trace=None, threshold=0.8):
    gold_text = str(example.answer)
    pred_text = str(pred.answer)

    # Get embeddings
    gold_emb = embedder([gold_text])[0]
    pred_emb = embedder([pred_text])[0]

    # Calculate similarity
    similarity = cosine_similarity(gold_emb, pred_emb)

    return similarity >= threshold


example = dspy.Example(
    answer="The study found that exercise improves mental health."
)
pred = dspy.Prediction(
    answer="Research shows physical activity boosts psychological well-being."
)

print(semantic_similarity_metric(example, pred))

## Multi-field similarity

If your signature has several output fields, compute similarity for each and average them. This rewards programs that get every field roughly right, not just one.

In [ ]:
def multi_field_semantic_metric(example, pred, trace=None):
    similarities = []

    for field in ["answer", "explanation", "confidence"]:
        gold_text = str(getattr(example, field, ""))
        pred_text = str(getattr(pred, field, ""))

        if gold_text and pred_text:
            gold_emb = embedder([gold_text])[0]
            pred_emb = embedder([pred_text])[0]
            sim = cosine_similarity(gold_emb, pred_emb)
            similarities.append(sim)

    return np.mean(similarities)


example = dspy.Example(
    answer="Paris",
    explanation="Paris is the capital and largest city of France.",
    confidence="high",
)
pred = dspy.Prediction(
    answer="Paris, France",
    explanation="Paris is France's capital and most populous city.",
    confidence="very high",
)

print(multi_field_semantic_metric(example, pred))

## Caching embeddings

Embeddings are deterministic — the same text and model always produce the same vector. That makes them perfectly cacheable. For serious workloads use a vector database; for local development a simple in-memory dict (or a pickle file) is enough.

In [ ]:
import hashlib
import pickle

embedding_cache = {}

def get_embedding_cached(text):
    # Hash text to use as cache key
    key = hashlib.md5(text.encode()).hexdigest()

    if key not in embedding_cache:
        embedding_cache[key] = embedder([text])[0]

    return embedding_cache[key]


# Warm the cache
v1 = get_embedding_cached("The cat sat on the mat")
# Second call returns the cached vector without an API request
v2 = get_embedding_cached("The cat sat on the mat")

print("cache size:", len(embedding_cache))
print("same vector returned:", v1 is v2)

# Persist the cache between runs
with open("embedding_cache.pkl", "wb") as f:
    pickle.dump(embedding_cache, f)

After the first run you only pay for *new* texts. For optimization workflows where gold answers are reused across many candidate prompts, this can cut embedding costs to near zero.